# 06: shrinkage vs rotation as a function of depth

Adaptive RMU layer sweep (`python -m rmu.layer_sweep`): split, words, β 1, lr 5e-4, α 5,
`--max_num_batches 257` (≈129 steps on the split data). Hooked layer L, trained
`mlp.down_proj.weight` in L−2..L, every layer started from the original weights and the same
control direction u. Each row of `sweep.csv` averages the last 30 steps.

With β = 1 and cos(a₀, u) ≈ 0:

    1 − L/L₀ ≈ (1 − k²)/2  [shrinkage]  +  k·cosθ  [rotation]

Copy the job's `sweep.csv` (`runs_root/<JOB_ID>/lapa-12b-instruct_layer-sweep/sweep.csv`) next to this notebook.

In [ ]:
from pathlib import Path

import pandas as pd

from rmu.plots import plot_layer_sweep

SWEEP = Path("sweep.csv")
sweep = pd.read_csv(SWEEP).sort_values("layer")
approx = sweep["forget_shrinkage"] + sweep["forget_rotation"]
sweep["forget_rotation_share"] = sweep["forget_rotation"] / approx
sweep["forget_approx_error"] = approx - sweep["forget_drop"]
sweep.round(4)

In [ ]:
fig = plot_layer_sweep(
    sweep,
    title="Adaptive RMU on Lapa-12B: shrinkage vs rotation per layer",
    save_to="06_layer_sweep.png",
)

**How to read it.** Yellow = shrinkage term (1 − k²)/2, violet = rotation term k·cosθ, so the bar
height ≈ relative drop 1 − L/L₀. Black dots = measured drop mean(1 − L/L₀); where a dot leaves the bar
top, the approximation (β = 1, cos(a₀, u) ≈ 0) breaks. Lighter, thinner bars = the same terms on retain
data; if they match the forget bars, the change is not selective.

Shrinkage is counted first. The split depends on that order, because the −2βk·cosθ term couples k and θ,
so treat the shares as approximate.